In [ ]:
#Problem Statement 1
def length_of_longest_substring_bruteforce(s: str) -> int:
    n = len(s)
    if n == 0:
        return 0

    longest_length = 0


    for i in range(n):

        for j in range(i, n):
            current_substring = s[i : j + 1]


            if len(set(current_substring)) == len(current_substring):
                longest_length = max(longest_length, len(current_substring))

    return longest_length

# Example usage:
input_str = "abcabcbb"
result = length_of_longest_substring_bruteforce(input_str)
print(f"Input: {input_str}")
print(f"Output: {result}")



Input: abcabcbb
Output: 3


In [ ]:
#Problem Statement 2
def product_except_self_bruteforce(nums: list[int]) -> list[int]:
    n = len(nums)
    answer = [0] * n

    for i in range(n):
        product = 1
        for j in range(n):
            if i != j:
                product *= nums[j]
        answer[i] = product
    return answer

# Example Usage:
input_nums_1 = [1, 2, 3, 4]
output_1 = product_except_self_bruteforce(input_nums_1)
print(f"Input: {input_nums_1}, Output: {output_1}") # Expected: [24, 12, 8, 6]



Input: [1, 2, 3, 4], Output: [24, 12, 8, 6]


In [ ]:
#Problem Statement 3
class LRUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}


        self.head = Node(0, 0)
        self.tail = Node(0, 0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove_node(self, node: Node):

        prev_node = node.prev
        next_node = node.next
        prev_node.next = next_node
        next_node.prev = prev_node

    def _add_to_front(self, node: Node):

        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _move_to_front(self, node: Node):

        self._remove_node(node)
        self._add_to_front(node)

    def get(self, key: int) -> int:
        if key in self.cache:
            node = self.cache[key]
            self._move_to_front(node)
            return node.value
        return -1

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self._move_to_front(node)
        else:
            new_node = Node(key, value)
            self.cache[key] = new_node
            self._add_to_front(new_node)

            if len(self.cache) > self.capacity:

                lru_node = self.tail.prev
                self._remove_node(lru_node)
                del self.cache[lru_node.key]

Let's test the `LRUCache` with an example.

In [ ]:
# Example Usage:

lru_cache = LRUCache(2)

print("--- Test Case 1: Basic Operations ---")
lru_cache.put(1, 1)
lru_cache.put(2, 2)
print(f"get(1): {lru_cache.get(1)}")

lru_cache.put(3, 3)
print(f"get(2): {lru_cache.get(2)}")

lru_cache.put(4, 4)
print(f"get(1): {lru_cache.get(1)}")
print(f"get(3): {lru_cache.get(3)}")
print(f"get(4): {lru_cache.get(4)}")

print("\n--- Test Case 2: Update Existing Key ---")
lru_cache_2 = LRUCache(2)
lru_cache_2.put(1, 1)
lru_cache_2.put(2, 2)
print(f"Initial Cache get(1): {lru_cache_2.get(1)}")
lru_cache_2.put(1, 100)
lru_cache_2.put(3, 3)
print(f"get(2) after eviction: {lru_cache_2.get(2)}")
print(f"get(1) after update: {lru_cache_2.get(1)}")
print(f"get(3): {lru_cache_2.get(3)}")

--- Test Case 1: Basic Operations ---
get(1): 1
get(2): -1
get(1): -1
get(3): 3
get(4): 4

--- Test Case 2: Update Existing Key ---
Initial Cache get(1): 1
get(2) after eviction: -1
get(1) after update: 100
get(3): 3


In [ ]:
!pip install flask pyngrok

In [ ]:
#Problem Statement 4
from flask import Flask, request, render_template_string, redirect
import sqlite3
import string

app = Flask(__name__)

BASE62 = string.digits + string.ascii_letters

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>URL Shortener</title>
</head>
<body>
    <h2>URL Shortener</h2>

    <form method="POST">
        <input type="text" name="url" placeholder="Enter Long URL" required style="width:400px">
        <button type="submit">Shorten</button>
    </form>

    {% if short_url %}
        <h3>Short URL:</h3>
        <a href="{{ short_url }}">{{ short_url }}</a>
    {% endif %}
</body>
</html>
"""

def init_db():
    conn = sqlite3.connect("urls.db")
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS urls(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        short_code TEXT UNIQUE,
        long_url TEXT
    )
    """)

    conn.commit()
    conn.close()

def encode(num):
    if num == 0:
        return BASE62[0]

    result = ""

    while num:
        result = BASE62[num % 62] + result
        num //= 62

    return result

@app.route("/", methods=["GET", "POST"])
def home():

    short_url = None

    if request.method == "POST":

        long_url = request.form["url"]

        conn = sqlite3.connect("urls.db")
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO urls(long_url) VALUES(?)",
            (long_url,)
        )

        row_id = cur.lastrowid

        short_code = encode(row_id)

        cur.execute(
            "UPDATE urls SET short_code=? WHERE id=?",
            (short_code, row_id)
        )

        conn.commit()
        conn.close()

        short_url = request.host_url + short_code

    return render_template_string(
        HTML,
        short_url=short_url
    )

@app.route("/<short_code>")
def redirect_to_url(short_code):

    conn = sqlite3.connect("urls.db")
    cur = conn.cursor()

    cur.execute(
        "SELECT long_url FROM urls WHERE short_code=?",
        (short_code,)
    )

    result = cur.fetchone()

    conn.close()

    if result:
        return redirect(result[0])

    return "URL Not Found", 404

if __name__ == "__main__":
    init_db()
    app.run(debug=True)

In [ ]:
#Problem Statement 5
import threading
import queue
import time
import random


buffer = queue.Queue(maxsize=5)

def producer(pid):
    for i in range(5):
        item = f"Item-{pid}-{i}"

        buffer.put(item)

        print(f"Producer {pid} produced {item}")

        time.sleep(random.uniform(0.5, 1.5))

def consumer(cid):
    while True:
        item = buffer.get()

        print(f"Consumer {cid} consumed {item}")

        time.sleep(random.uniform(0.5, 2))

        buffer.task_done()


producers = []
for i in range(2):
    t = threading.Thread(target=producer, args=(i + 1,))
    producers.append(t)


consumers = []
for i in range(2):
    t = threading.Thread(target=consumer, args=(i + 1,), daemon=True)
    consumers.append(t)


for t in producers:
    t.start()

for t in consumers:
    t.start()


for t in producers:
    t.join()


buffer.join()

print("All items processed successfully!")